# bronze → HTML レポート: 秋田県オープンデータの CSV を評価して可視化する

SageMaker Unified Studio の JupyterLab(Python 3 カーネル)で上から順に実行する。
bronze(`opendata_landing_to_bronze.ipynb` が書いた UTF-8・LF の CSV)を **Athena を通さず直接** 読む。
114 データセットは列の形がばらばらで、Glue のテーブル定義を1本ずつ作るより、
ファイルをそのまま読んで評価するほうが実態に合うため。

出すもの: 外部ファイルを一切参照しない HTML 1枚(図は SVG を埋め込み、JavaScript なし)。
公開する前提なので、HTML にはバケット名・アカウント ID を入れない。

| 出力 | 場所 |
| --- | --- |
| HTML | `s3://<bucket>/akita/reports/opendata/index.html` |
| 集計(JSON) | `s3://<bucket>/akita/reports/opendata/report.json` |

公開用のバケットへのコピーはこのノートブックの外で行う(実行ロールに公開先への権限を持たせないため)。

評価の観点は、デジタル庁「公開データの作成・掲載に関する手引き」/ 総務省
「統計表における機械判読可能なデータの表記方法の統一ルール」(2020)を参考にした、
**機械で読むときに手直しが要るか** の 12 項目。ルールそのものの適合判定ではない。

手元で試すとき(AWS に触らない): 環境変数 `OPENDATA_LOCAL_ROOT` にディレクトリを指定すると、
そこをバケットの代わりに読み書きする(`make opendata-local` が使う)。

出典: 秋田県オープンデータカタログサイト(https://ckan.pref.akita.lg.jp/), CC BY 4.0

In [ ]:
# ── 設定 ──────────────────────────────────────────────────────────────────
import os

# True のあいだは HTML を作って表示するだけで S3 には書かない。
DRY_RUN = os.environ.get("DRY_RUN", "1") != "0"

REGION = "ap-northeast-1"
BUCKET_PREFIX = "jaws-festa-2026"
BRONZE_PREFIX = "akita/bronze/opendata/"
MANIFEST_KEY = BRONZE_PREFIX + "_manifest.json"
OUTPUT_PREFIX = "akita/reports/opendata/"

# 手元にも同じものを書く(SageMaker 上ではノートブックの隣)。空にすると書かない
LOCAL_OUT = os.environ.get("OPENDATA_REPORT_OUT", "opendata_report")

LOCAL_ROOT = os.environ.get("OPENDATA_LOCAL_ROOT")

TITLE = "秋田県オープンデータ CSV の機械判読性レポート"

In [ ]:
# ── 読み書き先(S3 か、手元のディレクトリ) ──────────────────────────────────
from pathlib import Path


class S3Store:
    def __init__(self, bucket: str, client):
        self.bucket, self.s3 = bucket, client
        self.label = f"s3://{bucket}/"

    def get(self, key: str) -> bytes:
        return self.s3.get_object(Bucket=self.bucket, Key=key)["Body"].read()

    def put(self, key: str, body: bytes, content_type: str) -> None:
        self.s3.put_object(Bucket=self.bucket, Key=key, Body=body, ContentType=content_type)


class LocalStore:
    def __init__(self, root: str):
        self.root = Path(root).resolve()
        self.label = f"{self.root}/"

    def get(self, key: str) -> bytes:
        return (self.root / key).read_bytes()

    def put(self, key: str, body: bytes, content_type: str) -> None:
        p = self.root / key
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_bytes(body)


if LOCAL_ROOT:
    store = LocalStore(LOCAL_ROOT)
else:
    import boto3

    session = boto3.Session(profile_name=os.environ.get("AWS_PROFILE"), region_name=REGION)
    account_id = session.client("sts").get_caller_identity()["Account"]
    store = S3Store(f"{BUCKET_PREFIX}-{account_id}-{REGION}-an", session.client("s3"))
print("読み書き先:", store.label, "/ DRY_RUN =", DRY_RUN)

In [ ]:
import json
from concurrent.futures import ThreadPoolExecutor

manifest = json.loads(store.get(MANIFEST_KEY))
files = manifest["files"]
with ThreadPoolExecutor(max_workers=16) as ex:
    texts = list(ex.map(lambda f: store.get(BRONZE_PREFIX + f["file"]).decode("utf-8"), files))
print(f"bronze: {len(files)} 本 / {len(manifest['datasets'])} データセット"
      f"(台帳 {manifest['built_at']}, 取得 {manifest['landing_fetched_at']})")
print(f"取得の段階で落ちたもの: {len(manifest['not_fetched'])} 本 / bronze に入らなかったもの: {len(manifest['failed'])} 本")

## 1. ファイルごとの評価

見出しの位置は「空でないセルが、先頭50行で最も多い行の半分以上ある最初の行」とする。
その上にある行は表題・単位・空行(= 前置き)とみなす。

| # | 項目 | 手直しが要るとみなす条件 |
| --- | --- | --- |
| 1 | 文字コード | UTF-8 以外(取得時。bronze では UTF-8 にしてある) |
| 2 | 見出しの位置 | 1行目が見出しでない(表題・単位・空行が上にある) |
| 3 | 見出しの形 | 見出しが複数行 / 見出しに空欄がある(セル結合の名残) |
| 4 | 見出しの重複 | 同じ名前の列が2本以上ある |
| 5 | 列数 | 見出しと列数が違う行がある |
| 6 | 空行 | 表の途中に空行がある |
| 7 | 空の列 | 見出しも値も空の列がある(末尾のカンマなど) |
| 8 | 注釈行 | 表の下や途中に注・出典などの1セルだけの行がある |
| 9 | 数値の書き方 | 数値の列に桁区切り `,`・`%`・全角数字・前後の空白がある |
| 10 | 記号 | 数値の列に `－` `***` `△` などの記号が混ざる |
| 11 | 和暦 | 値に和暦(令和・平成・H30 など)がある |
| 12 | 横持ち | 年や日付が列名に並んでいる(年が増えるたびに列が増える形) |

点数は付けない。項目ごとの重みや「どの程度なら問題か」を決める根拠が無く、合計点にすると
「文字コードが読めない」と「和暦が1セルある」が同じ重さに見えてしまうため。
項目ごとに、当てはまるかどうかと、当てはまった箇所(行数・セル数など)だけを示す。

In [ ]:
import csv
import io
import math
import re
import statistics
import unicodedata
from collections import Counter

PLACEHOLDERS = {"-", "－", "―", "—", "‐", "−", "*", "**", "***", "…", "...", "x", "X", "･", "・", "ー", "‐", "*"}
NUM_RE = re.compile(r"^[+-]?(\d+(\.\d*)?|\.\d+)$")
WAREKI_RE = re.compile(r"(明治|大正|昭和|平成|令和)\s*(\d|元|[０-９一二三四五六七八九十])|^[MTSHR]\s?\d{1,2}[.年]")
YEAR_HEADER_RE = re.compile(r"((19|20)\d{2})\s*(年|年度)?|^(19|20)\d{2}[/-]\d{1,2}|(令和|平成|昭和)\s*\S+\s*年")
NOTE_RE = re.compile(r"^\s*[※＊*]|^\s*[（(]?\s*(注|出典|資料|備考)")


def parse_number(s: str) -> tuple[float | None, set[str]]:
    """数値として読めれば値を返す。読むために手直しした点を flags に入れる。"""
    flags: set[str] = set()
    t = s
    if t != t.strip():
        flags.add("空白")
        t = t.strip()
    n = unicodedata.normalize("NFKC", t)
    if n != t and any("０" <= ch <= "９" for ch in t):
        flags.add("全角数字")
    t = n.replace("−", "-")
    if t[:1] in ("△", "▲"):
        t = "-" + t[1:].strip()
        flags.add("△▲でマイナス")
    if "," in t:
        t2 = t.replace(",", "")
        if NUM_RE.match(t2):
            flags.add("桁区切り")
            t = t2
    if t.endswith("%") and NUM_RE.match(t[:-1].strip()):
        flags.add("%")
        t = t[:-1].strip()
    if NUM_RE.match(t):
        return float(t), flags
    return None, set()


def nonempty(row: list[str]) -> int:
    return sum(1 for c in row if c.strip())


def numeric_ratio(row: list[str]) -> float:
    cells = [c for c in row if c.strip()]
    return sum(1 for c in cells if parse_number(c)[0] is not None) / len(cells) if cells else 0.0


def find_header(rows: list[list[str]]) -> int:
    counts = [nonempty(r) for r in rows[:50]]
    w = max(counts, default=0)
    for i, n in enumerate(counts):
        if n >= max(1, math.ceil(w * 0.5)) and (n >= 2 or w <= 1):
            return i
    return 0


def analyze(f: dict, text: str) -> dict:
    rows = list(csv.reader(io.StringIO(text)))
    hdr = find_header(rows)
    header = rows[hdr] if rows else []
    ncols = len(header)
    w = nonempty(header)

    # 見出しの直後に数値を含まない行が続き、その先が数値の多い表なら、複数行の見出しとみなす
    body_start = hdr + 1
    later = [numeric_ratio(r) for r in rows[hdr + 1:hdr + 40] if nonempty(r) >= 2]
    med = statistics.median(later) if later else 0.0
    extra_header = 0
    while (body_start < len(rows) and extra_header < 3 and med >= 0.3
           and nonempty(rows[body_start]) >= 1 and numeric_ratio(rows[body_start]) == 0
           and nonempty(rows[body_start]) < w):
        body_start += 1
        extra_header += 1
    filled = [i for i, c in enumerate(header) if c.strip()]
    header_gaps = bool(filled) and any(not header[i].strip() for i in range(filled[0], filled[-1] + 1))

    # 列名: 複数行の見出しは上の行の空欄を左から埋めてから縦につなぐ
    names = []
    for j in range(ncols):
        parts = []
        for r in rows[hdr:body_start]:
            v = r[j].strip() if j < len(r) else ""
            if not v and r is rows[hdr]:
                v = next((r[k].strip() for k in range(j - 1, -1, -1) if k < len(r) and r[k].strip()), "")
            if v and v not in parts:
                parts.append(v)
        names.append(" / ".join(parts) or f"(空欄{j + 1})")
    names = [unicodedata.normalize("NFKC", n).replace("\n", " ") for n in names]

    body = rows[body_start:]
    # 末尾の注釈行と空行を表から外す(途中のものは数える)
    notes = 0
    while body and (nonempty(body[-1]) == 0 or (w >= 3 and nonempty(body[-1]) == 1
                                                 and parse_number(next(c for c in body[-1] if c.strip()))[0] is None)):
        notes += nonempty(body[-1]) == 1
        body.pop()
    inner_notes = sum(1 for r in body if r and NOTE_RE.match(r[0]) and nonempty(r) <= 2)
    blank_rows = sum(1 for r in body if nonempty(r) == 0)
    data = [r for r in body if nonempty(r) > 0 and not (r and NOTE_RE.match(r[0]) and nonempty(r) <= 2)]
    ragged = sum(1 for r in data if len(r) != ncols)

    cols = []
    for j in range(ncols):
        vals = [r[j] if j < len(r) else "" for r in data]
        cells = [v for v in vals if v.strip()]
        ph = [v for v in cells if v.strip() in PLACEHOLDERS]
        real = [v for v in cells if v.strip() not in PLACEHOLDERS]
        parsed = [parse_number(v) for v in real]
        nums = [p for p, _ in parsed if p is not None]
        kind = "数値" if real and len(nums) >= 0.9 * len(real) else ("空" if not cells else "文字")
        flags = Counter(fl for p, fls in parsed if p is not None for fl in fls) if kind == "数値" else Counter()
        cols.append({
            "name": names[j],
            "kind": kind,
            "empty_ratio": 1 - len(cells) / len(vals) if vals else 1.0,
            "placeholders": len(ph) if kind == "数値" else 0,
            "format_flags": dict(flags),
            "distinct": len(set(cells)),
            "wareki": sum(1 for v in cells if WAREKI_RE.search(v)),
        })

    empty_cols = sum(1 for j, c in enumerate(cols) if c["kind"] == "空" and not header[j].strip())
    year_headers = sum(1 for c in header if YEAR_HEADER_RE.search(unicodedata.normalize("NFKC", c)))
    dup = [n for n, k in Counter(n for n in names if not n.startswith("(空欄")).items() if k > 1]
    fmt = Counter()
    for c in cols:
        fmt.update(c["format_flags"])
    ph_total = sum(c["placeholders"] for c in cols)
    wareki = sum(c["wareki"] for c in cols)

    checks = {
        "encoding": (f["encoding_in"].startswith("UTF-8"), f["encoding_in"]),
        "preamble": (hdr == 0, f"見出しは {hdr + 1} 行目" if hdr else ""),
        "header_shape": (not extra_header and not header_gaps,
                         "; ".join(x for x in (f"見出し {extra_header + 1} 行" if extra_header else "",
                                               "見出しに空欄" if header_gaps else "") if x)),
        "dup_header": (not dup, ", ".join(dup[:3])),
        "ragged": (ragged == 0, f"{ragged} 行" if ragged else ""),
        "blank_rows": (blank_rows == 0, f"{blank_rows} 行" if blank_rows else ""),
        "empty_cols": (empty_cols == 0, f"{empty_cols} 列" if empty_cols else ""),
        "notes": (notes + inner_notes == 0, f"{notes + inner_notes} 行" if notes + inner_notes else ""),
        "number_format": (not fmt, ", ".join(f"{k} {v}" for k, v in fmt.most_common())),
        "placeholders": (ph_total == 0, f"{ph_total} セル" if ph_total else ""),
        "wareki": (wareki == 0, f"{wareki} セル" if wareki else ""),
        "wide": (year_headers < 3, f"年・日付の列 {year_headers} 本" if year_headers >= 3 else ""),
    }
    return {
        **{k: f[k] for k in ("file", "dataset", "resource_name", "last_modified", "encoding_in", "newline_in",
                             "bytes_in")},
        "header_row": hdr,
        "header_rows": extra_header + 1,
        "names": names,
        "raw_header": header,
        "data": data,
        "columns": cols,
        "n_rows": len(data),
        "n_cols": ncols,
        "checks": checks,
    }


evals = [analyze(f, t) for f, t in zip(files, texts)]
CHECK_LABELS = {
    "encoding": "文字コードが UTF-8 以外", "preamble": "1行目が見出しでない", "header_shape": "見出しが複数行・空欄あり",
    "dup_header": "見出しの重複", "ragged": "列数がそろわない行", "blank_rows": "途中に空行",
    "empty_cols": "空の列", "notes": "注釈行", "number_format": "数値に桁区切り・%・全角",
    "placeholders": "数値の列に記号", "wareki": "和暦", "wide": "横持ち(年が列に並ぶ)",
}
fail_counts = Counter(k for e in evals for k, (ok, _) in e["checks"].items() if not ok)
CHECK_SHORT = {
    "encoding": "文字コード", "preamble": "見出しの位置", "header_shape": "見出しの形", "dup_header": "見出しの重複",
    "ragged": "列数", "blank_rows": "空行", "empty_cols": "空の列", "notes": "注釈行", "number_format": "数値の書き方",
    "placeholders": "記号", "wareki": "和暦", "wide": "横持ち",
}
no_issue = sum(1 for e in evals if all(ok for ok, _ in e["checks"].values()))
print(f"評価したファイル: {len(evals)} 本 / どの項目にも当てはまらないもの {no_issue} 本")
for k, label in CHECK_LABELS.items():
    print(f"  {label:<20} {fail_counts[k]:>4} 本")

## 2. 年度などで分かれたファイルの見出しは同じか

年度ごとにファイルが分かれているものは、まとめて1つのテーブルとして読みたくなる。
そのとき見出しがファイルによって違うと、列を **位置** で読む素の構成(S3 に CSV + Athena)では
別の列の値が入る。特に **列数が同じで名前だけ違う** 場合は、列数の検査もすり抜ける。

1つのデータセットに別々の表(施設の種類ごとなど)が入っていることも多いので、
**リソース名から年・年度・数字を除くと同じ名前になるもの** を「同じ表の年違い(系列)」とみなす。
例: `県・25市町村の人口・世帯数（2014(平成26)年）` と `…（2015(平成27)年）`。

見出しは全角・半角と空白をそろえ、年・年度を表す部分(`2019年度` → `#年度`)を伏せてから比べる。
「列名に年度が入っているだけで意味は同じ」ものを変化に数えないため。

In [ ]:
YEAR_TOKEN = re.compile(r"(19|20)\d{2}|(令和|平成|昭和)\s*(\d+|元)")


def series_key(e: dict) -> str:
    s = unicodedata.normalize("NFKC", e["resource_name"] or e["file"].split("/", 1)[1][9:])
    s = re.sub(r"(令和|平成|昭和)\s*(\d+|元)\s*年?度?", "", s)
    s = re.sub(r"\d+", "", s)
    return re.sub(r"[年度月日分～~\-_()（）\s・、,]|現在|時点|公表|公開|時系列|速報|統計版|版", "", s)


def header_key(e: dict) -> tuple[str, ...]:
    return tuple(YEAR_TOKEN.sub("#", re.sub(r"\s+", "", n)) for n in e["names"])


def classify(base: tuple, other: tuple) -> str:
    if other == base:
        return "同じ"
    if len(other) != len(base):
        return "列数が変わる"
    if sorted(other) == sorted(base):
        return "順序だけ変わる"
    return "列数は同じで名前が変わる"


by_ds: dict[str, list[dict]] = {}
groups: dict[tuple[str, str], list[dict]] = {}
for e in evals:
    by_ds.setdefault(e["dataset"], []).append(e)
    groups.setdefault((e["dataset"], series_key(e)), []).append(e)

drift = []
for (ds, _), es in groups.items():
    if len(es) < 2:
        continue
    es = sorted(es, key=lambda e: e["last_modified"] or "")
    keys = Counter(header_key(e) for e in es)
    base = keys.most_common(1)[0][0]
    kinds = Counter(classify(base, header_key(e)) for e in es)
    drift.append({
        "dataset": ds,
        "series": es[-1]["resource_name"] or es[-1]["file"],
        "files": len(es),
        "variants": len(keys),
        "kinds": dict(kinds),
        "ncols": sorted({len(k) for k in keys}),
        "examples": [
            {"file": e["resource_name"] or e["file"], "kind": classify(base, header_key(e)), "ncols": e["n_cols"],
             "added": [n for n, k in zip(e["names"], header_key(e)) if k not in base][:3],
             "removed": [n for n in base if n not in header_key(e)][:3]}
            for e in es if header_key(e) != base
        ][:6],
    })
drift.sort(key=lambda d: (-d["variants"], -d["files"]))
changed = [d for d in drift if d["variants"] > 1]
# 「列数は同じ」ファイルが1本でもある系列 = 列数の検査では気づけない
silent = [d for d in changed if d["kinds"].get("列数は同じで名前が変わる") or d["kinds"].get("順序だけ変わる")]
print(f"年度などで分かれた系列: {len(drift)}({sum(d['files'] for d in drift)} 本 / "
      f"{len({d['dataset'] for d in drift})} データセット)")
print(f"  うち見出しが変わる: {len(changed)} / うち列数が同じまま名前・順序が変わるファイルを含む: {len(silent)}")
for d in changed[:15]:
    print(f"  {d['dataset']:<30} {d['series'][:24]:<26} {d['files']:>3} 本 {d['variants']:>2} 通り {d['kinds']}")

## 3. 図を作る

図は SVG を文字列で組み立てる(matplotlib を使わない)。文字はブラウザのフォントで描かれるので、
SageMaker に日本語フォントが無くても日本語が化けない。値・列名は CSV 由来なので、
全部 `html.escape` してから埋め込む(公開ページに CSV の中身がそのまま入るため)。

データセットごとの図は、代表の1本(更新日が最も新しいもの)から自動で選ぶ。

| 条件 | 図 |
| --- | --- |
| 緯度・経度の列がある | 位置の散布図 |
| 年・年月日の列があり、数値の列がある | 時系列(同じ年に複数行あれば、1行目と同じ区分の行だけ) |
| 年・年月日の列があり、数値の列が無い | 年(月)ごとの件数 |
| 列名に年が3つ以上並ぶ(横持ち) | 先頭の行を年ごとの折れ線に |
| 上のどれでもない | 区分の列の件数(上位12) |

In [ ]:
import html

PALETTE = ["#1f6feb", "#d1242f", "#1a7f37", "#8250df", "#bf8700"]
esc = html.escape


def nice_ticks(lo: float, hi: float, n: int = 5) -> list[float]:
    if hi == lo:
        hi = lo + 1
    raw = (hi - lo) / n
    mag = 10 ** math.floor(math.log10(raw))
    step = next(m * mag for m in (1, 2, 2.5, 5, 10) if m * mag >= raw)
    start = math.floor(lo / step) * step
    ticks = [start]
    while ticks[-1] < hi:  # 最後の目盛りが最大値以上になるまで(図の上端で線が切れないように)
        ticks.append(ticks[-1] + step)
    return ticks


def fmt_num(v: float) -> str:
    if abs(v) >= 1e8:
        return f"{v / 1e8:g}億"
    if abs(v) >= 1e4:
        return f"{v / 1e4:g}万"
    return f"{v:,.0f}" if float(v).is_integer() else f"{v:,.2f}"


def svg_open(w: int, h: int, label: str) -> str:
    return (f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {w} {h}" width="100%" role="img" '
            f'aria-label="{esc(label)}" style="max-width:{w}px;font-family:inherit"><title>{esc(label)}</title>')


def line_chart(series: dict[str, list[tuple[float, float]]], label: str, w: int = 680, h: int = 260,
               x_is_year: bool = True) -> str:
    pts = [p for s in series.values() for p in s]
    if not pts:
        return ""
    l, r, t, b = 64, 16, 14, 46 + 16 * ((len(series) + 2) // 3)
    xs, ys = [p[0] for p in pts], [p[1] for p in pts]
    x0, x1 = min(xs), max(xs)
    yt = nice_ticks(min(0, min(ys)), max(ys))
    y0, y1 = yt[0], yt[-1]
    X = lambda x: l + (x - x0) / ((x1 - x0) or 1) * (w - l - r)  # noqa: E731
    Y = lambda y: h - b - (y - y0) / ((y1 - y0) or 1) * (h - t - b)  # noqa: E731
    out = [svg_open(w, h, label)]
    for v in yt:
        out.append(f'<line x1="{l}" x2="{w - r}" y1="{Y(v):.1f}" y2="{Y(v):.1f}" stroke="#ddd"/>'
                   f'<text x="{l - 6}" y="{Y(v) + 4:.1f}" font-size="11" text-anchor="end" fill="#555">{esc(fmt_num(v))}</text>')
    xt = nice_ticks(x0, x1, 6)
    if x_is_year:  # 年の目盛りは整数だけにする(2021, 2021.5 … が「2021」と2回並ぶのを避ける)
        step = max(1, math.ceil((x1 - x0) / 6))
        xt = list(range(math.ceil(x0), math.floor(x1) + 1, step))
    for v in xt:
        if x0 <= v <= x1:
            out.append(f'<text x="{X(v):.1f}" y="{h - b + 16}" font-size="11" text-anchor="middle" fill="#555">'
                       f'{esc(str(int(v)) if x_is_year else fmt_num(v))}</text>')
    for i, (name, s) in enumerate(series.items()):
        c = PALETTE[i % len(PALETTE)]
        s = sorted(s)
        d = " ".join(f"{X(x):.1f},{Y(y):.1f}" for x, y in s)
        out.append(f'<polyline fill="none" stroke="{c}" stroke-width="1.8" points="{d}"/>')
        if len(s) <= 40:
            out += [f'<circle cx="{X(x):.1f}" cy="{Y(y):.1f}" r="2.4" fill="{c}"/>' for x, y in s]
        lx, ly = l + (i % 3) * ((w - l) // 3), h - b + 34 + 16 * (i // 3)
        out.append(f'<rect x="{lx}" y="{ly - 9}" width="10" height="10" fill="{c}"/>'
                   f'<text x="{lx + 14}" y="{ly}" font-size="11" fill="#333">{esc(name[:28])}</text>')
    out.append("</svg>")
    return "".join(out)


def bar_chart(items: list[tuple[str, float]], label: str, w: int = 680, unit: str = "",
              colors: list[str] | None = None) -> str:
    if not items:
        return ""
    bh, l, r = 20, 190, 70
    h = bh * len(items) + 10
    m = max(v for _, v in items) or 1
    out = [svg_open(w, h, label)]
    for i, (name, v) in enumerate(items):
        y = 5 + i * bh
        bw = v / m * (w - l - r)
        c = (colors or [PALETTE[0]])[i % len(colors or [PALETTE[0]])]
        out.append(f'<text x="{l - 6}" y="{y + 14}" font-size="12" text-anchor="end" fill="#333">{esc(name[:22])}</text>'
                   f'<rect x="{l}" y="{y + 3}" width="{bw:.1f}" height="{bh - 6}" fill="{c}"/>'
                   f'<text x="{l + bw + 4:.1f}" y="{y + 14}" font-size="12" fill="#333">{esc(fmt_num(v))}{esc(unit)}</text>')
    out.append("</svg>")
    return "".join(out)


def scatter(points: list[tuple[float, float]], label: str, w: int = 420, h: int = 440) -> str:
    if len(points) > 3000:  # 点が多すぎると HTML が重くなるので間引く
        step = len(points) / 3000
        points = [points[int(i * step)] for i in range(3000)]
    xs, ys = [p[0] for p in points], [p[1] for p in points]
    pad = 24
    x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys)
    k = min((w - 2 * pad) / ((x1 - x0) or 0.01), (h - 2 * pad) / ((y1 - y0) or 0.01))
    out = [svg_open(w, h, label), f'<rect x="0" y="0" width="{w}" height="{h}" fill="#f6f8fa"/>']
    out += [f'<circle cx="{pad + (x - x0) * k:.1f}" cy="{h - pad - (y - y0) * k:.1f}" r="2" fill="{PALETTE[1]}" '
            f'fill-opacity="0.45"/>' for x, y in points]
    out.append(f'<text x="{pad}" y="{h - 6}" font-size="11" fill="#555">経度 {x0:.2f}〜{x1:.2f} / 緯度 {y0:.2f}〜{y1:.2f}'
               f'(北が上・{len(points):,} 点)</text></svg>')
    return "".join(out)

In [ ]:
# ── データセットごとに図を1枚選ぶ ─────────────────────────────────────────────
TIME_NAME = re.compile(r"年月日|年月|年度|(^|/ )年$|西暦|日時|日付|^年[＼\\/ ]|^月$|時点|date", re.I)
YEAR_ONLY = re.compile(r"\s*((?:19|20)\d{2})\s*(年|年度)?\s*(実績|末|計)?\s*")
TOTAL_LABEL = re.compile(r"^(合計|総数|総計|計|県計|全県|秋田県計?|市部計|郡部計|全体|小計)$")
CODE_NAME = re.compile(r"コード|番号|^no\.?$|^№|^id$|_id|郵便|電話|fax|^番$", re.I)
CATEGORY_HINT = re.compile(r"市町村|地域|振興局|種別|分類|区分|類型|設置者|獣種|業種|部局")


def parse_time(v: str) -> float | None:
    v = unicodedata.normalize("NFKC", v).strip()
    if re.fullmatch(r"\d{10}", v):  # 年月日コード(YYYY00MMDD)
        y, m = int(v[:4]), int(v[6:8])
        return y + (m - 1) / 12 if 1 <= m <= 12 else float(y)
    m = re.match(r"^((?:19|20)\d{2})\s*[-/年.]\s*(\d{1,2})", v)
    if m and 1 <= int(m.group(2)) <= 12:
        return int(m.group(1)) + (int(m.group(2)) - 1) / 12
    m = re.fullmatch(r"((?:19|20)\d{2})\s*(年|年度)?\s*(末)?", v)
    if m:
        return float(m.group(1))
    return None


def number(v: str) -> float | None:
    return None if v.strip() in PLACEHOLDERS else parse_number(v)[0]


def chart_for(e: dict) -> tuple[str, str]:
    names, data, cols = e["names"], e["data"], e["columns"]
    col = lambda j: [r[j] if j < len(r) else "" for r in data]  # noqa: E731

    # 横持ち(列名が年だけのものが3本以上)。1行でも折れ線にできるので最初に見る
    ycols = [(j, YEAR_ONLY.fullmatch(unicodedata.normalize("NFKC", h))) for j, h in enumerate(e["raw_header"])]
    ycols = [(j, float(m.group(1))) for j, m in ycols if m]
    if len(ycols) >= 3 and len({x for _, x in ycols}) == len(ycols):
        series = {}
        for r in data[:12]:
            lab = next((c.strip() for c in r if c.strip() and number(c) is None), "") or names[0]
            pts = [(x, number(r[j])) for j, x in ycols if j < len(r)]
            pts = [(x, y) for x, y in pts if y is not None]
            if len(pts) >= 3:
                series.setdefault(lab, pts)
            if len(series) == 4:
                break
        if series:
            return line_chart(series, f"{e['dataset']} 横持ちの年"), "列名に並んだ年を横軸にした先頭の行"
    if len(data) < 2:
        return "", "行が少ないので図は無し"

    # 緯度・経度
    lat = next((j for j, n in enumerate(names) if "緯度" in n or n.lower() in ("lat", "latitude")), None)
    lon = next((j for j, n in enumerate(names) if "経度" in n or n.lower() in ("lon", "lng", "longitude")), None)
    if lat is not None and lon is not None:
        pts = [(number(r[lon]), number(r[lat])) for r in data if max(lat, lon) < len(r)]
        pts = [(x, y) for x, y in pts if x and y]
        # 秋田県のおおよその範囲(緯度 38.8〜40.6 / 経度 139.5〜141.1)の外は描かない。県外の所在地や
        # 緯度・経度の取り違えが1点あるだけで、残りが点の塊に潰れるため
        inside = [(x, y) for x, y in pts if 139.5 < x < 141.1 and 38.8 < y < 40.6]
        if len(inside) >= 5:
            out = len(pts) - len(inside)
            return (scatter(inside, f"{e['dataset']} 位置"),
                    f"「{names[lat]}」「{names[lon]}」の位置" + (f"(県の範囲外の {out} 点は描いていない)" if out else ""))

    nums = [j for j, c in enumerate(cols) if c["kind"] == "数値" and not CODE_NAME.search(names[j])]
    # 時系列
    tcols = [j for j, n in enumerate(names) if TIME_NAME.search(n) and "年齢" not in n]
    for tj in tcols:
        ts = [parse_time(v) for v in col(tj)]
        if sum(t is not None for t in ts) < 0.8 * len(ts) or len({t for t in ts if t}) < 3:
            continue
        ys = [j for j in nums if j != tj]
        if not ys:
            cnt = Counter(math.floor(t) for t in ts if t is not None)
            pts = sorted((float(k), float(v)) for k, v in cnt.items())
            return line_chart({"件数": pts}, f"{e['dataset']} 年ごとの件数"), f"「{names[tj]}」の年ごとの行数"
        texts = [j for j, c in enumerate(cols) if c["kind"] == "文字" and j != tj and not TIME_NAME.search(names[j])]
        first = data[0]
        keep = data
        desc = ""
        if len({t for t in ts if t is not None}) < len([t for t in ts if t is not None]):
            keep = [r for r in data if all((r[j] if j < len(r) else "") == first[j] for j in texts)]
            desc = "(" + " / ".join(first[j].strip() for j in texts[:3] if first[j].strip()) + " の行)" if texts else ""
        base = None
        series = {}
        for j in ys:
            pts = [(parse_time(r[tj]), number(r[j])) for r in keep if j < len(r)]
            pts = [(x, y) for x, y in pts if x is not None and y is not None]
            if len(pts) < 3:
                continue
            mx = max(abs(y) for _, y in pts) or 1
            base = base or mx
            if base / 20 <= mx <= base * 20:  # 桁が大きく違う列は同じ図に載せない
                series[names[j]] = pts
            if len(series) == 4:
                break
        if series:
            return (line_chart(series, f"{e['dataset']} 時系列"),
                    f"「{names[tj]}」を横軸にした数値の列{desc}")
    # 区分の件数
    cands = [j for j, c in enumerate(cols)
             if c["kind"] == "文字" and 2 <= c["distinct"] <= min(40, max(3, len(data) // 2))
             and not CODE_NAME.search(names[j])]
    if cands and len(data) >= 10:
        j = sorted(cands, key=lambda j: (not CATEGORY_HINT.search(names[j]), j))[0]
        top = Counter(v.strip() for v in col(j) if v.strip()).most_common(12)
        return bar_chart([(k, float(v)) for k, v in top], f"{e['dataset']} {names[j]}", unit=" 件"), \
            f"「{names[j]}」ごとの行数(上位12)"
    # 見出しの列 + 数値の列(行ごとの値の棒)。合計の行は外す
    labs = [j for j, c in enumerate(cols) if c["kind"] == "文字" and c["distinct"] >= 0.8 * len(data)
            and not CODE_NAME.search(names[j])]
    if labs and nums and len(data) >= 3:
        lj, vj = labs[0], nums[0]
        items = [(r[lj].strip(), number(r[vj])) for r in data if max(lj, vj) < len(r)]
        items = [(k, v) for k, v in items if k and v is not None and not TOTAL_LABEL.match(re.sub(r"\s", "", k))][:15]
        if len(items) >= 3:
            return bar_chart(items, f"{e['dataset']} {names[vj]}"), f"「{names[lj]}」ごとの「{names[vj]}」(先頭15行・合計の行を除く)"
    return "", "自動で選べる図が無い"


ds_meta = manifest["datasets"]
main_of = {}
for ds, es in by_ds.items():
    main_of[ds] = sorted(es, key=lambda e: (e["last_modified"] or "", e["n_rows"]))[-1]
charts = {ds: chart_for(e) for ds, e in main_of.items()}
print(f"図を作れたデータセット: {sum(1 for c, _ in charts.values() if c)} / {len(charts)}")
print(Counter(d.split("」")[-1] if "」" in d else d for _, d in charts.values()).most_common(8))

## 4. HTML を組み立てる

1枚に、全体の要約 → 項目ごとの不備の数 → 見出しの変化 → データセットごとの評価と図、の順で並べる。
データセットごとの節は `<details>` で畳んでおく(114 個あるため)。
CC BY 4.0 の表示として、出典・ライセンス・加工したことを冒頭と末尾に書く。

In [ ]:
from datetime import datetime, timedelta, timezone

JST = timezone(timedelta(hours=9))
now = datetime.now(JST).strftime("%Y-%m-%d %H:%M JST")
enc = Counter(e["encoding_in"] for e in evals)


def safe_url(u: str | None) -> str:
    return esc(u) if u and u.startswith("https://ckan.pref.akita.lg.jp/") else ""


def table(head: list[str], rows: list[list[str]], cls: str = "") -> str:
    """head と rows は HTML として埋め込み済みの文字列を受け取る(呼ぶ側で esc 済み)。"""
    th = "".join(f"<th>{h}</th>" for h in head)
    tr = "".join("<tr>" + "".join(f"<td>{c}</td>" for c in r) + "</tr>" for r in rows)
    return f'<table class="{cls}"><thead><tr>{th}</tr></thead><tbody>{tr}</tbody></table>'


def hits(es: list[dict], k: str) -> int:
    return sum(1 for e in es if not e["checks"][k][0])


def tags(keys: list[str]) -> str:
    return " ".join(f'<span class="tag">{esc(CHECK_SHORT[k])}</span>' for k in keys) or '<span class="sub">該当なし</span>'


check_chart = bar_chart([(CHECK_LABELS[k], float(v)) for k, v in fail_counts.most_common()],
                        "項目ごとの手直しが要るファイル数", unit=" 本", colors=[PALETTE[1]])
enc_chart = bar_chart([(k, float(v)) for k, v in enc.most_common()], "取得時の文字コード", unit=" 本")
drift_kinds = Counter()
for d in drift:
    if d["variants"] == 1:
        drift_kinds["見出しが全部同じ"] += 1
    elif d in silent:
        drift_kinds["列数は同じで列名が変わる"] += 1
    else:
        drift_kinds["列数が変わる"] += 1
drift_chart = bar_chart([(k, float(drift_kinds[k])) for k in ("見出しが全部同じ", "列数が変わる", "列数は同じで列名が変わる")
                         if drift_kinds[k]], "年度などで分かれた表の見出し", unit=" 系列",
                        colors=[PALETTE[2], PALETTE[4], PALETTE[1]])


drift_rows = []
for d in changed:
    ex = "<br>".join(
        f"{esc(x['file'][:40])}: {esc(x['kind'])}(列 {x['ncols']})"
        + (f" 増「{esc('」「'.join(a[:16] for a in x['added']))}」" if x["added"] else "")
        + (f" 減「{esc('」「'.join(a[:16] for a in x['removed']))}」" if x["removed"] else "")
        for x in d["examples"][:3])
    title = (ds_meta.get(d["dataset"]) or {}).get("dataset_title", d["dataset"])
    drift_rows.append([f'<a href="#{esc(d["dataset"])}">{esc(title)}</a><br><small>{esc(d["series"][:50])}</small>',
                       str(d["files"]), str(d["variants"]), esc(" / ".join(map(str, d["ncols"]))), ex])

index_rows = []
sections = []
drift_ds = {d["dataset"] for d in changed}
ds_order = sorted(by_ds, key=lambda ds: ((ds_meta.get(ds) or {}).get("organization") or "", ds))
for ds in ds_order:
    es, meta = by_ds[ds], ds_meta.get(ds) or {}
    title = meta.get("dataset_title", ds)
    cells = []
    for k in CHECK_LABELS:
        n = hits(es, k)
        cells.append(f'<span class="hit">{"●" if len(es) == 1 else n}</span>' if n else "")
    cells.append('<span class="hit">●</span>' if ds in drift_ds else "")
    index_rows.append([f'<a href="#{esc(ds)}">{esc(title)}</a>', esc(meta.get("organization") or ""), str(len(es)),
                       *cells])
    applied = [k for k in CHECK_LABELS if hits(es, k)]
    svg, how = charts[ds]
    m = main_of[ds]
    preview_cols = list(range(min(len(m["names"]), 10)))
    preview = table([esc(m["names"][j][:24]) for j in preview_cols],
                    [[esc((r[j] if j < len(r) else "")[:30]) for j in preview_cols] for r in m["data"][:3]], "preview")
    frows = []
    for e in sorted(es, key=lambda e: e["file"]):
        issues = "<br>".join(f"{esc(CHECK_LABELS[k])}" + (f"({esc(v[:60])})" if v else "")
                             for k, (ok, v) in e["checks"].items() if not ok) or "—"
        frows.append([esc(e["resource_name"] or e["file"]), esc(e["encoding_in"]), f"{e['n_rows']:,}", str(e["n_cols"]),
                      issues])
    dnote = "".join(
        f'<p class="warn">「{esc(d["series"][:40])}」の {d["files"]} 本は、年度などで見出しが {d["variants"]} 通りある。</p>'
        for d in changed if d["dataset"] == ds)
    link = safe_url(meta.get("dataset_url"))
    sections.append(f"""
<details id="{esc(ds)}"><summary>{esc(title)} <small>{esc(meta.get('organization') or '')}・CSV {len(es)} 本</small><br>{tags(applied)}</summary>
<p>出典: {f'<a href="{link}" rel="noopener">{esc(title)}</a>' if link else esc(title)}(秋田県, CC BY 4.0)</p>
{dnote}
<h4>図: {esc(how)}</h4>
<p class="sub">代表のファイル: {esc(m['resource_name'] or m['file'])}</p>
{svg or '<p class="sub">(図は無し)</p>'}
<h4>先頭3行(代表のファイル。見出しは評価で読み取ったもの)</h4>
<div class="scroll">{preview}</div>
<h4>ファイルごとの評価</h4>
<div class="scroll">{table(["ファイル", "文字コード", "行", "列", "当てはまる項目(箇所)"], frows)}</div>
</details>""")

not_fetched = Counter(f"{'中身が ' + (x.get('detected') or '') if x['status'] == 'not_csv' else '大きすぎて取得していない'}"
                      for x in manifest["not_fetched"])
not_fetched_ds = sorted({x["dataset"] for x in manifest["not_fetched"]})

CSS = """
body{font-family:-apple-system,BlinkMacSystemFont,"Hiragino Sans","Noto Sans JP","Yu Gothic",Meiryo,sans-serif;
max-width:980px;margin:0 auto;padding:24px 16px 64px;color:#1f2328;line-height:1.65}
h1{font-size:1.6rem;margin:.2em 0}h2{border-bottom:1px solid #d0d7de;padding-bottom:.2em;margin-top:2.2em}
h4{margin:1.2em 0 .3em}.sub,small{color:#59636e}a{color:#0969da}
.kpi{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:12px;margin:16px 0}
.kpi div{background:#f6f8fa;border:1px solid #d0d7de;border-radius:8px;padding:10px 14px}
.kpi b{display:block;font-size:1.6rem}
table{border-collapse:collapse;font-size:.85rem;width:100%}th,td{border:1px solid #d0d7de;padding:4px 8px;
vertical-align:top;text-align:left}th{background:#f6f8fa}.scroll{overflow-x:auto}
table.preview td,table.preview th{white-space:nowrap}
.tag{display:inline-block;font-size:.75rem;font-weight:400;border:1px solid #d1242f;color:#d1242f;border-radius:10px;
padding:0 7px;margin:2px 2px 0 0}
table.matrix th.v{writing-mode:vertical-rl;white-space:nowrap;font-size:.75rem;padding:6px 2px}
table.matrix td{text-align:center}table.matrix td:first-child,table.matrix td:nth-child(2){text-align:left}
.hit{color:#d1242f;font-weight:700}
details{border:1px solid #d0d7de;border-radius:8px;padding:6px 12px;margin:8px 0}
summary{cursor:pointer;font-weight:600}.warn{background:#fff8c5;border:1px solid #d4a72c;padding:6px 10px;border-radius:6px}
footer{margin-top:3em;font-size:.85rem;color:#59636e;border-top:1px solid #d0d7de;padding-top:1em}
"""

page = f"""<!doctype html>
<html lang="ja"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<meta http-equiv="Content-Security-Policy" content="default-src 'none'; style-src 'unsafe-inline'; img-src data:">
<title>{esc(TITLE)}</title><style>{CSS}</style></head><body>
<h1>{esc(TITLE)}</h1>
<p class="sub">作成 {esc(now)} / データ取得 {esc(manifest['landing_fetched_at'])}(UTC)</p>
<p>秋田県オープンデータカタログサイトで CSV 形式として公開されている全データセットを取得し、
<b>機械で読むときに手直しが要るか</b> を 12 項目で調べた。データの正しさや価値ではなく、プログラムでそのまま読めるかどうかを見ている。
項目ごとの重みを決める根拠が無いので、合計の点数は付けていない。</p>
<p class="sub">出典: <a href="https://ckan.pref.akita.lg.jp/" rel="noopener">秋田県オープンデータカタログサイト</a>(秋田県)、
<a href="https://creativecommons.org/licenses/by/4.0/deed.ja" rel="license noopener">CC BY 4.0</a>。
このページは同サイトのデータを加工(文字コード・改行の変換、集計、図の作成)して作成したもので、秋田県が作成したものではない。</p>

<div class="kpi">
<div><b>{len(by_ds)}</b>データセット</div>
<div><b>{len(evals)}</b>CSV ファイル</div>
<div><b>{no_issue}</b>どの項目にも当てはまらないファイル</div>
<div><b>{len(changed)} / {len(drift)}</b>年度などで分かれた表のうち見出しが変わるもの</div>
</div>

<h2>1. 何が手直しを要したか</h2>
<p>項目ごとに、手直しが要るファイルの数。1本のファイルが複数の項目に当てはまる。</p>
{check_chart}
<h4>取得時の文字コード</h4>
<p class="sub">bronze ではすべて UTF-8(BOMなし)・改行 LF にそろえてある。取得時の改行は全ファイル CRLF だった。</p>
{enc_chart}

<h2>2. 年度ごとのファイルをまとめて読むと何が起きるか</h2>
<p>同じ表が年度ごとに別の CSV になっているもの(リソース名から年・数字を除くと同じ名前になるもの)を
{len(drift)} 系列・{sum(d['files'] for d in drift)} 本見つけた。これらを S3 に置いて1つのテーブルとして読むと、
列は名前ではなく <b>位置</b> で対応づけられる。見出しが途中で変わると別の列の値が入り、
<b>列数が同じなら列数の検査もすり抜ける</b>(エラーにならない)。
列名に入った年(<code>2019年度実績</code> など)は伏せてから比べている。</p>
{drift_chart}
<div class="scroll">{table(["データセット / 系列", "CSV", "見出しの種類", "列数", "多数派と違うファイルの例"], drift_rows)}</div>

<h2>3. データセットごとの評価と図</h2>
<p>担当部署の順。表の数字は、その項目に当てはまる CSV の本数(CSV が1本だけのデータセットは ●)。
「見出しの変化」は 2. で見出しが変わると分かった系列を含むもの。
各データセットの節の図は代表の1本から自動で選んだもので、内容を解釈したものではない。</p>
<div class="scroll">{table(["データセット", "担当", "CSV"] + [f'<span>{esc(v)}</span>' for v in CHECK_SHORT.values()] + ["見出しの変化"], index_rows, "matrix").replace("<th><span>", '<th class="v"><span>').replace("<th>見出しの変化", '<th class="v">見出しの変化')}</div>
{''.join(sections)}

<h2>評価の対象にしていないもの</h2>
<p>{esc('、'.join(f'{k} {v} 本' for k, v in not_fetched.items()))}
({len(not_fetched_ds)} データセット)。形式は CSV と登録されているが中身が XLSX のもの、
1本 50MB を超えるもの(地震被害想定の格子データ)は取得の段階で外した。
{f'文字コードなどの理由で bronze に入らなかったもの {len(manifest["failed"])} 本。' if manifest['failed'] else ''}</p>

<footer>
<p>{esc(TITLE)} / 出典: 秋田県オープンデータカタログサイト(https://ckan.pref.akita.lg.jp/)、CC BY 4.0。
本ページは出典のデータを加工して作成した。評価の観点は デジタル庁「公開データの作成・掲載に関する手引き」、
総務省「統計表における機械判読可能なデータの表記方法の統一ルール」を参考にした独自のもので、適合判定ではない。</p>
<p>JAWS FESTA 2026 in AKITA「CSV置くだけ構成は壊れずに間違う ― 秋田のオープンデータで検証するS3 Tables」の検証の一部として作成。</p>
</footer>
</body></html>
"""
body = page.encode("utf-8")
print(f"HTML: {len(body) / 1e6:.2f} MB")

In [ ]:
# ── 公開してはいけない情報が入っていないか(バケット名・アカウント ID) ─────────
leaks = [s for s in {getattr(store, "bucket", None), globals().get("account_id"), str(getattr(store, "root", ""))}
         if s and s in page]
assert not leaks, f"HTML に公開してはいけない文字列が入っている: {leaks}"
# 外部を読み込む要素が無いこと(自己完結)
assert not re.search(r"<(script|link|iframe|img)\b", page, re.I), "外部を読み込む要素がある"
print("公開前の確認 OK")

In [ ]:
# ── 書き出す ────────────────────────────────────────────────────────────────
summary = {
    "generated_at": now,
    "landing_fetched_at": manifest["landing_fetched_at"],
    "datasets": len(by_ds),
    "files": len(evals),
    "files_no_issue": no_issue,
    "check_fail_counts": {k: fail_counts[k] for k in CHECK_LABELS},
    "encoding_in": dict(enc),
    "series": len(drift),
    "header_changes": len(changed),
    "header_changes_same_ncols": len(silent),
    "drift": drift,
    "files_detail": [
        {k: e[k] for k in ("file", "dataset", "encoding_in", "header_row", "header_rows", "n_rows", "n_cols")}
        | {"failed": [k for k, (ok, _) in e["checks"].items() if not ok]}
        for e in evals
    ],
}
sbody = json.dumps(summary, ensure_ascii=False, indent=1).encode("utf-8")

if LOCAL_OUT:
    Path(LOCAL_OUT).mkdir(parents=True, exist_ok=True)
    Path(LOCAL_OUT, "index.html").write_bytes(body)
    Path(LOCAL_OUT, "report.json").write_bytes(sbody)
    print(f"手元: {Path(LOCAL_OUT).resolve()}/index.html")
if DRY_RUN:
    print(f"[dry-run] {OUTPUT_PREFIX}index.html と report.json は書かない")
else:
    store.put(OUTPUT_PREFIX + "index.html", body, "text/html; charset=utf-8")
    store.put(OUTPUT_PREFIX + "report.json", sbody, "application/json")
    print(f"書いた: {store.label}{OUTPUT_PREFIX}index.html")

In [ ]:
# ノートブック上で見る(JupyterLab ではこのセルの出力にレポートが表示される)
from IPython.display import HTML, display

display(HTML(f'<iframe srcdoc="{esc(page)}" style="width:100%;height:800px;border:1px solid #ccc"></iframe>'))